In [1]:
import pandas as pd
import numpy as np
import os
from coniferest.pineforest import PineForest
from coniferest.session import Session
from coniferest.session.callback import (TerminateAfter, prompt_decision_callback,)
from coniferest.session.callback import (TerminateAfter, viewer_decision_callback,)
from coniferest.label import Label

In [2]:
from tqdm import tqdm

def load_single(oid_filename, feature_filename):
    oid     = np.memmap(oid_filename, mode='c', dtype=np.uint64)
    feature = np.memmap(feature_filename, mode='c', dtype=np.float32).reshape(oid.shape[0], -1)
    return oid, feature

In [3]:
fields = [686, 769]

In [4]:
all_oids = []
all_features = []

for field in fields:
    oid_filename = f"/media2/SNAD/dr23-features-collected_by_field/dr23_oid_{field}.dat"
    feature_filename = f"/media2/SNAD/dr23-features-collected_by_field/dr23_feature_{field}.dat"

    oids_field, features_field = load_single(oid_filename, feature_filename)

    all_oids.append(np.array(oids_field))
    all_features.append(np.array(features_field))

oids_use = np.concatenate(all_oids)
features_use = np.concatenate(all_features)
oids_use


array([686201100000000, 686201100000001, 686201100000002, ...,
       769216400076366, 769216400076368, 769216400076369],
      shape=(13635400,), dtype=uint64)

In [5]:
data2 = pd.read_csv("/media/tomy/git/canidates_log - Sheet5.csv", header = None)
data2.columns = ["url"]
data_art = data2["url"].str.split("/").str[-1]
    

In [6]:
data = pd.read_parquet('/media/tomy/git/top_5000_iter_006.parquet')
pd.set_option('display.max_rows', None)
data_2500 = data.iloc[0:2500]
OIDs = []

for o, inp in enumerate (data_2500['id'], start=0):
    oid_part, mjd_part = inp.split('_')
    oid = int(oid_part.removeprefix('ZTFDR'))
    OIDs.append((o, oid))

In [7]:
fields = ["686", "769"]

In [8]:
labels_coniferest = []
g_field = []

for i in OIDs:
    idx, oid = i
    field = str(oid)[:3]
    if field in fields :
        g_field.append(oid)
        if str(oid) in data_art.values:
            labels_coniferest.append(Label.REGULAR)
        else:
            labels_coniferest.append(Label.ANOMALY)

#for i in labels_coniferest :
    #print (i)

#print (len(labels_coniferest))
#for h in g_field :             
   #print (h)

In [9]:
final_features = []
final_labels = []
final_oids = []

oids_use_str = oids_use.astype(str)

for oid, label in zip(g_field, labels_coniferest):
    oid_str = str(oid)
    match = np.where(oids_use_str == oid_str)[0]
    if len(match) > 0:
        idx = match[0]
        final_features.append(features_use[idx])
        final_labels.append(label)
        final_oids.append(oid)

final_features = np.array(final_features)
final_labels = np.array(final_labels)
final_oids = np.array(final_oids)

In [10]:
model_ztf = PineForest(random_seed=42)

model_ztf.fit_known(
    features_use,
    known_data=final_features,
    known_labels=final_labels
)

In [11]:
data = features_use
metadata = oids_use

In [15]:
model_ztf = PineForest(42)

class RecordCallback:
    def __init__(self):
        self.records = []

    def __call__(self, metadata, data, session):
        decision = session.last_decision
        
        if decision == 1:
            label = "REGULAR"
        elif decision == -1:
            label = "ANOMALY"
        else:
            label = "UNKNOWN"
        self.records.append(f'{metadata} -> {label}')

    def print_report(self):
        print('Records:')
        print('\n'.join(self.records))
        
record_callback = RecordCallback()

session_ztf = Session(
    data=data,
    metadata=metadata,
    model=model_ztf,
    # Prompt for a decision and open object's page on the SNAD Viewer
    decision_callback=viewer_decision_callback,
    on_decision_callbacks=[
        record_callback,
        TerminateAfter(2),
    ]
)
session_ztf.run()
record_callback.print_report()

Check https://ztf.snad.space/view/769201400038351 for details
Is 769201400038351 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Check https://ztf.snad.space/view/686216300067840 for details
Is 686216300067840 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Records:
769201400038351 -> REGULAR
686216300067840 -> REGULAR


In [17]:
df_records = pd.DataFrame(record_callback.records,columns=["Report"])
df_records.to_csv("/media/tomy/git/report_callback_no_priors.csv", index=False)